# DeepGold RL — Quickstart

End-to-end walkthrough: **data → features → environment → short PPO training → out-of-sample backtest**.

Run this from the project root (or it will auto-add the root to `sys.path`). Works in both Jupyter and the VSCode notebook editor.

> The default data is **synthetic** — replace `data/XAUUSD_M5.csv` with a real MT5 export for genuine research.

In [ ]:
import sys, os
from pathlib import Path

# Make the project importable regardless of where the kernel started.
ROOT = Path.cwd()
if (ROOT / 'config').exists() is False and (ROOT.parent / 'config').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(name)s | %(message)s')

from config.config import Config

cfg = Config.from_yaml('config/config.yaml')
cfg.paths.ensure()
cfg.to_dict()['env']

## 1. Generate synthetic data (skip if you have real data in `data/`)

In [ ]:
from utils.sample_data import write_sample_csv

csv_path = cfg.paths.data / f'{cfg.data.symbol}_{cfg.data.timeframe}.csv'
if not csv_path.exists():
    write_sample_csv(csv_path, timeframe=cfg.data.timeframe, start='2021-01-01', end='2025-12-31')
print('Data file:', csv_path)

## 2. Build the leakage-safe pipeline (load → features → split → normalize)

The normalizer is fit on the **training** set only and applied to 2025.

In [ ]:
from env.env_builder import TradingDataPipeline

pipeline = TradingDataPipeline(cfg)
train_df, test_df = pipeline.prepare()
print('Train bars:', len(train_df), ' Test (2025) bars:', len(test_df))
print('Features:', pipeline.feature_columns)
train_df[pipeline.feature_columns].describe().T

## 3. Sanity-check the environment with a random policy

In [ ]:
env = pipeline.make_env('train', random_start=False)
obs, info = env.reset()
print('Observation shape:', obs.shape, '| Action space:', env.action_space)

done = False
steps = 0
while not done and steps < 2000:
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    done = terminated or truncated
    steps += 1
print(f'Random policy ran {steps} steps; final equity = {info["equity"]:.2f}; trades = {info["n_trades"]}')

## 4. Short PPO training run

Use a small number of timesteps here just to see the pipeline learn; use `scripts/train.py` for full runs.

In [ ]:
from training.train_ppo import PPOTrainer, resolve_device
print('Device:', resolve_device(cfg.training.device))

cfg.training.model_name = 'ppo_gold_nb'
cfg.training.n_envs = 2
trainer = PPOTrainer(cfg, pipeline=pipeline)
trainer.train(total_timesteps=20_000)
model_path = trainer.save()
model_path

## 5. Backtest on the held-out 2025 data

In [ ]:
from backtest.backtester import Backtester

bt = Backtester(cfg, pipeline=pipeline)
report = bt.run(model_name='ppo_gold_nb', on='test')
print(report.pretty())

In [ ]:
%matplotlib inline
from utils import visualization as viz

hist = bt.history
_ = viz.plot_equity_curve(hist['equity_curve'], timestamps=hist['timestamps'],
                          initial_balance=hist['initial_balance'],
                          title='Out-of-sample Equity Curve (2025)')

In [ ]:
_ = viz.plot_reward_curve(hist['rewards'], title='Reward over Test Episode')

## Next steps

- Replace synthetic data with a real `XAUUSD_M5.csv` export from MetaTrader 5.
- Run a full training job: `python scripts/train.py`.
- Tune `config/config.yaml` (reward weights, risk limits, PPO hyper-parameters).
- Inspect TensorBoard: `tensorboard --logdir logs/tensorboard`.
- Wire up live trading via `live_trading/` once you trust the backtests.